# Execution Plan Validation
Demonstrate the race condition fix and plan validation

In [ ]:
import asyncio
import json
from derivatives_gpt_core.agents.pricing.nodes.create_execution_plan import validate_and_fix_execution_plan
from derivatives_gpt_core.schemas.llm_schemas import ExecutionPlan, Task
from utils.common import timer

## Create Invalid Plan (Race Condition)

In [ ]:
# Create an invalid plan where fetch and pricing are in same group
invalid_plan = ExecutionPlan(
    tasks=[
        Task(id="fetch_spot", type="market_data", description="Fetch AAPL spot price"),
        Task(id="fetch_vol", type="volatility", description="Calculate volatility"),
        Task(id="fetch_rate", type="risk_free_rate", description="Get risk-free rate"),
        Task(id="price_option", type="pricing", description="Price the option")
    ],
    parallel_groups=[
        ["fetch_spot", "fetch_vol", "fetch_rate", "price_option"]  # INVALID: All in same group!
    ],
    can_execute=True
)

print("INVALID PLAN (Before Fix):")
print("=" * 40)
print("Parallel Groups:")
for i, group in enumerate(invalid_plan.parallel_groups):
    print(f"  Group {i}: {group}")
print("\n⚠️ Problem: Pricing task in same group as fetch tasks!")

## Apply Validation and Auto-Fix

In [ ]:
@timer
def fix_plan(plan):
    validate_and_fix_execution_plan(plan)
    return plan

fixed_plan = fix_plan(invalid_plan)

print("FIXED PLAN (After Validation):")
print("=" * 40)
print("Parallel Groups:")
for i, group in enumerate(fixed_plan.parallel_groups):
    print(f"  Group {i}: {group}")
print("\n✅ Fixed: Fetch tasks execute before pricing task!")

## Complex Multi-Leg Plan

In [ ]:
# Iron Condor with invalid grouping
complex_plan = ExecutionPlan(
    tasks=[
        Task(id="fetch_spy", type="market_data", description="Fetch SPY data"),
        Task(id="fetch_vol", type="volatility", description="Get implied vol"),
        Task(id="price_put1", type="pricing", description="Price put 440"),
        Task(id="price_put2", type="pricing", description="Price put 445"),
        Task(id="price_call1", type="pricing", description="Price call 455"),
        Task(id="price_call2", type="pricing", description="Price call 460"),
        Task(id="aggregate", type="aggregation", description="Calculate net premium")
    ],
    parallel_groups=[
        ["fetch_spy", "fetch_vol", "price_put1", "price_put2"],  # INVALID!
        ["price_call1", "price_call2"],
        ["aggregate"]
    ],
    can_execute=True
)

print("COMPLEX PLAN (Before):")
for i, group in enumerate(complex_plan.parallel_groups):
    print(f"  Group {i}: {group}")

validate_and_fix_execution_plan(complex_plan)

print("\nCOMPLEX PLAN (Fixed):")
for i, group in enumerate(complex_plan.parallel_groups):
    print(f"  Group {i}: {group}")

## Valid Plan (No Changes Needed)

In [ ]:
# Already valid plan
valid_plan = ExecutionPlan(
    tasks=[
        Task(id="fetch_data", type="market_data", description="Fetch market data"),
        Task(id="price_opt1", type="pricing", description="Price option 1"),
        Task(id="price_opt2", type="pricing", description="Price option 2")
    ],
    parallel_groups=[
        ["fetch_data"],  # Group 1: Fetch
        ["price_opt1", "price_opt2"]  # Group 2: Price in parallel
    ],
    can_execute=True
)

original_groups = valid_plan.parallel_groups.copy()
validate_and_fix_execution_plan(valid_plan)

print("Valid Plan Groups:")
for i, group in enumerate(valid_plan.parallel_groups):
    print(f"  Group {i}: {group}")

if original_groups == valid_plan.parallel_groups:
    print("\n✅ Plan was already valid - no changes needed")
else:
    print("\n⚠️ Plan was modified")

## Performance Impact

In [ ]:
# Simulate execution times
import time

def simulate_execution(groups, fetch_time=0.2, price_time=0.1):
    """Simulate execution time for parallel groups."""
    total_time = 0
    for group in groups:
        # Each group runs in parallel, so time = max(task times)
        group_time = 0
        for task in group:
            if 'fetch' in task or 'market' in task or 'vol' in task:
                group_time = max(group_time, fetch_time)
            elif 'price' in task:
                group_time = max(group_time, price_time)
            else:
                group_time = max(group_time, 0.05)
        total_time += group_time
    return total_time

# Invalid plan (if it could run)
invalid_time = simulate_execution([["fetch_spot", "fetch_vol", "price_option"]])

# Fixed plan
fixed_time = simulate_execution([["fetch_spot", "fetch_vol"], ["price_option"]])

print("Execution Time Comparison:")
print("=" * 40)
print(f"Invalid plan (would fail): {invalid_time*1000:.0f}ms")
print(f"Fixed plan (succeeds): {fixed_time*1000:.0f}ms")
print(f"\nNote: Invalid plan would actually fail with race condition!")